### cluster (only for train set)

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import os

DATASET = "Pancreas"
DATA_PATH = f"dataset/{DATASET}/{DATASET}.h5ad"
SAVE_PATH = f"dataset/{DATASET}/{DATASET}_gene_clusters.csv"
N_GENES = 2000
N_CLUSTERS = 64  # as the numbers of token 

def cluster():
    print(f"Loading data from {DATA_PATH}...")
    adata = sc.read(DATA_PATH)
    
    if hasattr(adata.X, "toarray"):
        X = adata.X.toarray()
    else:
        X = adata.X
    
    # gene relation
    print("Computing gene correlation matrix...")
    corr_matrix = np.corrcoef(X, rowvar=False)
    corr_matrix = np.nan_to_num(corr_matrix)
    
    # K-Means 
    print(f"Running K-Means (k={N_CLUSTERS})...")
    kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    gene_labels = kmeans.fit_predict(corr_matrix)
    
    # svae
    df = pd.DataFrame({
        "gene_index": np.arange(len(gene_labels)),
        "cluster_id": gene_labels
    })
    
    df.to_csv(SAVE_PATH, index=False)
    print(f"✅ Clustering complete. Labels saved to {SAVE_PATH}")

if __name__ == "__main__":
    cluster()

[Step 1] Loading data from scale/Pancreas_10.h5ad...
Computing gene correlation matrix...


/home/liyilang/.conda/envs/lyl/lib/python3.8/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/liyilang/.conda/envs/lyl/lib/python3.8/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Running K-Means (k=64)...
✅ Clustering complete. Labels saved to scale/Pancreas_10_gene_clusters.csv


### PCA

#### train set

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import joblib
import os

DATASET = "Pancreas"
DATA_PATH = f"dataset/{DATASET}/{DATASET}.h5ad" 
CLUSTER_PATH = f"dataset/{DATASET}/{DATASET}_gene_clusters.csv"
OUTPUT_DIR = f"dataset/{DATASET}"
CELL_TYPE_COLUMN = 'cell_type' 
PCA_DIM = 16  

def pca_train():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    file_name = os.path.basename(DATA_PATH)
    dataset_name = os.path.splitext(file_name)[0]
    print(f"🔹 Processing Training Dataset: {dataset_name}")

    if not os.path.exists(CLUSTER_PATH):
        print(f"❌ Error: can't find cluster file: {CLUSTER_PATH}")
        return

    # 1. load data
    print(f"Loading data: {DATA_PATH}")
    adata = sc.read(DATA_PATH)
    gene_clusters = pd.read_csv(CLUSTER_PATH)
    

    original_labels = adata.obs[CELL_TYPE_COLUMN].astype(str)
    n_before = len(original_labels.unique())
    
    # some same cell types might have diffrent names in differnent datasets
    rename_map = {}
    cleaned_labels = original_labels.replace(rename_map)
    adata.obs[CELL_TYPE_COLUMN] = cleaned_labels
    
    n_after = len(cleaned_labels.unique())
    print(f"   Done. Categories reduced from {n_before} to {n_after}.")
    print(f"   Final Categories: {sorted(cleaned_labels.unique())}")

    # 2. Label Mapping
    print("Generating Label Mapping...")
    
    le = LabelEncoder()
    int_labels = le.fit_transform(cleaned_labels)
    
    # { 'B cell': 0, 'T cell': 1 ... }
    label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"   Generated Label Schema: {label_mapping}")

    # 3. init matrix
    if hasattr(adata.X, "toarray"):
        X_data = adata.X.toarray()
    else:
        X_data = adata.X
    
    # 4. PCA 
    print("Learning Latent Space (PCA Fit_Transform)...")
    
    all_latent_tokens = []
    pca_models = {} 
    
    n_clusters = gene_clusters['cluster_id'].nunique()
    
    for cluster_id in range(n_clusters):
        current_cluster_genes = gene_clusters[gene_clusters['cluster_id'] == cluster_id]
        gene_indices = current_cluster_genes['gene_index'].values
        X_subset = X_data[:, gene_indices]
        n_features = X_subset.shape[1]
        n_components_actual = min(PCA_DIM, n_features)
        pca = PCA(n_components=n_components_actual)
        X_encoded = pca.fit_transform(X_subset)
        
        # save model
        pca_models[cluster_id] = pca
        if X_encoded.shape[1] < PCA_DIM:
            padding_size = PCA_DIM - X_encoded.shape[1]
            padding = np.zeros((X_encoded.shape[0], padding_size))
            X_encoded = np.hstack([X_encoded, padding])
            
        all_latent_tokens.append(X_encoded)
        
        if cluster_id % 10 == 0:
            print(f"   Processed cluster {cluster_id}/{n_clusters}", end='\r')

    final_latent = np.stack(all_latent_tokens, axis=1)
    
    # 5. save
    save_prefix = os.path.join(OUTPUT_DIR, dataset_name)
    path_tokens = f"{save_prefix}_latent_tokens.npy"
    path_labels = f"{save_prefix}_labels.npy"
    path_mapping = f"{save_prefix}_label_mapping.npy"
    path_pca_pkl = f"{save_prefix}_pca_models.pkl"
    
    np.save(path_tokens, final_latent)
    np.save(path_labels, int_labels)
    np.save(path_mapping, label_mapping)
    joblib.dump(pca_models, path_pca_pkl)
    
    print(f"\n✅ Training Process complete for {dataset_name}.")
    print(f"   Latent shape: {final_latent.shape}")
    print(f"   Label shape:  {int_labels.shape}")
    print("-" * 30)
    print(f"   📂 for test set:")
    print(f"     - {path_mapping}")
    print(f"     - {path_pca_pkl}")
    print("-" * 30)

if __name__ == "__main__":
    pca_train()

🔹 Processing Training Dataset: LungB
Loading data: dataset/LungB/LungB.h5ad
🧹 Cleaning up inconsistent labels (Merging synonyms)...
   Done. Categories reduced from 13 to 13.
   Final Categories: ['B cell', 'Ciliated', 'Endothelium', 'Fibroblast', 'Lymphatic', 'Macrophages', 'Mast cell', 'NK cell', 'Secretory', 'T cell', 'Transformed epithelium', 'Type 1', 'Type 2']
Generating Label Mapping...
   Generated Label Schema: {'B cell': 0, 'Ciliated': 1, 'Endothelium': 2, 'Fibroblast': 3, 'Lymphatic': 4, 'Macrophages': 5, 'Mast cell': 6, 'NK cell': 7, 'Secretory': 8, 'T cell': 9, 'Transformed epithelium': 10, 'Type 1': 11, 'Type 2': 12}
Learning Latent Space (PCA Fit_Transform)...
   Processed cluster 60/64
✅ Training Process complete for LungB.
   Latent shape: (8288, 64, 16)
   Label shape:  (8288,)
------------------------------
   📂 工件 (供测试集使用):
     - dataset/LungB/LungB_label_mapping.npy
     - dataset/LungB/LungB_pca_models.pkl
------------------------------


#### test set

In [6]:
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import joblib
import os

DATA_NAMES = ["Baron", "Xin", "Segerstolpe"]

TRAIN_NAME = "Pancreas"
OUTPUT_DIR = f"test/{TRAIN_NAME}"
CELL_TYPE_COLUMN = 'cell_type' 
# must use the corresponding train set
TRAIN_DATA_PATH = f"dataset/{TRAIN_NAME}/{TRAIN_NAME}.h5ad"                     
TRAIN_CLUSTER_PATH = f"dataset/{TRAIN_NAME}/{TRAIN_NAME}_gene_clusters.csv"    
TRAIN_PCA_PATH = f"dataset/{TRAIN_NAME}/{TRAIN_NAME}_pca_models.pkl"            
TRAIN_MAPPING_PATH = f"dataset/{TRAIN_NAME}/{TRAIN_NAME}_label_mapping.npy"    

def pca():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    file_name = os.path.basename(DATA_PATH)
    dataset_name = os.path.splitext(file_name)[0]
    print(f"🔹 Processing Test Dataset: {dataset_name}")

    required_files = [DATA_PATH, TRAIN_DATA_PATH, TRAIN_CLUSTER_PATH, TRAIN_PCA_PATH, TRAIN_MAPPING_PATH]
    for f in required_files:
        if not os.path.exists(f):
            print(f"❌ Error: can't find the file: {f}")
            return

    print("Loading training artifacts (Clusters, PCAs, Mappings)...")
    pca_models = joblib.load(TRAIN_PCA_PATH)
    gene_clusters = pd.read_csv(TRAIN_CLUSTER_PATH)
    
    train_mapping = np.load(TRAIN_MAPPING_PATH, allow_pickle=True).item()
    if isinstance(list(train_mapping.keys())[0], int):
        str_to_int_map = {v: k for k, v in train_mapping.items()}
    else:
        str_to_int_map = train_mapping
        
    print(f"   Training Label Schema: {str_to_int_map}")

    adata_train = sc.read(TRAIN_DATA_PATH)
    train_genes = adata_train.var_names
    print(f"   Standard Gene Order: {len(train_genes)} genes")

    print(f"Loading test data: {DATA_PATH}")
    adata_test = sc.read(DATA_PATH)
    
    print("Aligning genes to training standard...")
    if hasattr(adata_test.X, "toarray"):
        X_test_df = pd.DataFrame(adata_test.X.toarray(), columns=adata_test.var_names)
    else:
        X_test_df = pd.DataFrame(adata_test.X, columns=adata_test.var_names)
    
    X_test_aligned = X_test_df.reindex(columns=train_genes, fill_value=0).values
    print(f"   Aligned Shape: {X_test_aligned.shape}")

    print("Aligning labels to training standard...")
    if CELL_TYPE_COLUMN not in adata_test.obs.columns:
        print(f"❌ Error: clomn '{CELL_TYPE_COLUMN}' not exists")
        return

    raw_labels = adata_test.obs[CELL_TYPE_COLUMN].values
    
    int_labels = []
    valid_indices = []
    
    for i, label in enumerate(raw_labels):
        if label in str_to_int_map:
            int_labels.append(str_to_int_map[label])
            valid_indices.append(i)
        else:
            pass 
            
    int_labels = np.array(int_labels)
    X_test_aligned = X_test_aligned[valid_indices]
    
    if len(raw_labels) != len(int_labels):
        print(f"⚠️ Warning: Dropped {len(raw_labels) - len(int_labels)} cells due to unseen labels.")

    # PCA 
    print("Projecting into Latent Space (PCA Transform)...")
    all_latent_tokens = []
    n_clusters = gene_clusters['cluster_id'].nunique()
    PCA_DIM = 16 
    
    for cluster_id in range(n_clusters):
        gene_indices = gene_clusters[gene_clusters['cluster_id'] == cluster_id]['gene_index'].values
        
        X_subset = X_test_aligned[:, gene_indices]
        
        pca = pca_models[cluster_id]
        
        X_encoded = pca.transform(X_subset)
        
        if X_encoded.shape[1] < PCA_DIM:
            padding = np.zeros((X_encoded.shape[0], PCA_DIM - X_encoded.shape[1]))
            X_encoded = np.hstack([X_encoded, padding])
            
        all_latent_tokens.append(X_encoded)
    
    final_latent = np.stack(all_latent_tokens, axis=1) # (N, Clusters, 16)
    
    # save
    save_prefix = os.path.join(OUTPUT_DIR, dataset_name)
    path_tokens = f"{save_prefix}_latent_tokens.npy"
    path_labels = f"{save_prefix}_labels.npy"

    np.save(path_tokens, final_latent)
    np.save(path_labels, int_labels)
    
    print(f"✅ Process complete for {dataset_name}.")
    print(f"   Latent shape: {final_latent.shape}")
    print(f"   Labels shape: {int_labels.shape}")
    print(f"   Saved to:")
    print(f"     - {path_tokens}")
    print(f"     - {path_labels}")

if __name__ == "__main__":
    for name in DATA_NAMES:
        DATA_NAME = name
        DATA_PATH = f"test/{TRAIN_NAME}/{DATA_NAME}.h5ad"
        pca()

🔹 Processing Test Dataset: Baron
Loading training artifacts (Clusters, PCAs, Mappings)...
   Training Label Schema: {'MHC class II': 0, 'acinar': 1, 'alpha': 2, 'beta': 3, 'co-expression': 4, 'delta': 5, 'ductal': 6, 'endothelial': 7, 'epsilon': 8, 'gamma': 9, 'macrophage': 10, 'mast': 11, 'schwann': 12, 'stellate': 13, 't_cell': 14, 'unclassified': 15}
   Standard Gene Order: 2000 genes
Loading test data: test/Pancreas/Baron.h5ad
Aligning genes to training standard...
   Aligned Shape: (1714, 2000)
Aligning labels to training standard...
Projecting into Latent Space (PCA Transform)...
✅ Process complete for Baron.
   Latent shape: (1714, 64, 16)
   Labels shape: (1714,)
   Saved to:
     - test/Pancreas\Baron_latent_tokens.npy
     - test/Pancreas\Baron_labels.npy
🔹 Processing Test Dataset: Xin
Loading training artifacts (Clusters, PCAs, Mappings)...
   Training Label Schema: {'MHC class II': 0, 'acinar': 1, 'alpha': 2, 'beta': 3, 'co-expression': 4, 'delta': 5, 'ductal': 6, 'endothel